In [1]:
import os
os.chdir('../')
!ls

Notebooks README.md env.yml   model


In [2]:
import argparse
import json
from tqdm.notebook import tqdm

from PIL import Image
import torch
import torch.nn as nn
from torchvision.transforms.functional import to_pil_image
from huggingface_hub import hf_hub_download

import timm
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
import pickle

from model import *

In [3]:
model = timm.create_model("hf-hub:MahmoodLab/uni", pretrained=True, init_values=1e-5, dynamic_img_size=True)
transform = create_transform(**resolve_data_config(model.pretrained_cfg, model=model))
# model.eval()

In [4]:
sample = torch.zeros((1,3,448,448))
with torch.inference_mode():
    print(model(sample).shape) #im guessing this is just the CLS token embedding 

torch.Size([1, 1024])


In [5]:
from torchsummary import summary
summary(model, input_size=(3, 224, 224))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 1024, 14, 14]         787,456
          Identity-2         [-1, 14, 14, 1024]               0
        PatchEmbed-3         [-1, 14, 14, 1024]               0
           Dropout-4            [-1, 197, 1024]               0
          Identity-5            [-1, 197, 1024]               0
          Identity-6            [-1, 197, 1024]               0
         LayerNorm-7            [-1, 197, 1024]           2,048
            Linear-8            [-1, 197, 3072]       3,148,800
          Identity-9          [-1, 16, 197, 64]               0
         Identity-10          [-1, 16, 197, 64]               0
         Identity-11            [-1, 197, 1024]               0
           Linear-12            [-1, 197, 1024]       1,049,600
          Dropout-13            [-1, 197, 1024]               0
        Attention-14            [-1, 19

In [ ]:
from model import ABMIL

mil_batch = torch.zeros((128, 3, 224, 224)) #128 instances of 224x224xrgb

class UniABMIL(nn.Module):
    def __init__(self, encoder_args = None, emb_sz = 1024):
        super().__init__()
        self.encoder = timm.create_model("hf-hub:MahmoodLab/uni", pretrained=True, init_values=1e-5, dynamic_img_size=True)
        self.transform = create_transform(**resolve_data_config(model.pretrained_cfg, model=model))

        self.abmil = ABMIL(emb_sz = emb_sz, hidden_sz=1024)

        self.class_head = nn.Sequential(
            nn.Linear(1024, 512),
            nn.LeakyReLU(),
            nn.Linear(512, 4)
        )

    def forward(self, x):
        x = self.transform(x)
        x = self.encoder(x)

        x = self.abmil(x)
        x = self.class_head(x)
        return x

uniabmil = UniABMIL()
uniabmil(mil_batch)


In [ ]:
def pad_batch(batch, pad_batch_size):
    orig_len = len(batch)
    if len(batch) != pad_batch_size:
        _ = torch.zeros((pad_batch_size - len(batch),) + batch.shape[1:]).type(batch.type())
        batch = torch.vstack([batch, _])

    return batch, orig_len

In [ ]:
def embed_patches(model, dataloader):
    all_embs = []
    device = next(model.parameters())[0].device

    for batch in dataloader:
        batch = pad_batch(batch, dataloader.batch_size)

        batch, orig_len = batch.to(device)
        with torch.inference_mode():
            embeddings = model(batch).detach().cpu()[:orig_len, :].cpu()
        
        assert not torch.isnan(embeddings).any()

    all_embs.append(embeddings)
